<a href="https://colab.research.google.com/github/aligreo/TriEncoder-Unet-Project/blob/main/mslesseg_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!uv pip install SimpleITK monai nibabel

Using Python 3.12.13 environment at: /usr
Resolved 33 packages in 302ms
Prepared 2 packages in 1.24s
Installed 2 packages in 36ms
 + monai==1.5.2
 + simpleitk==2.5.4


In [28]:
import os
import random
import warnings
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    NormalizeIntensityd,
    ConcatItemsd,
    DeleteItemsd,
    EnsureTyped,
    CropForegroundd,
    Lambdad,
)
from monai.data import PersistentDataset, DataLoader

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [29]:
mslesseg_train_data = "/content/drive/MyDrive/mslesseg folder/MSLesSeg Dataset/train"
mslesseg_test_data = "/content/drive/MyDrive/mslesseg folder/MSLesSeg Dataset/test"

In [58]:
processed_path = "/content/drive/MyDrive/mslesseg folder/MSLesSeg-2024-main.zip"

# Unzip the file to the current directory
!unzip -q "{processed_path}" -d /content/

# List the contents to verify the extraction
!ls -F /content/MSLesSeg-2024-main/

README.md  registration.py  resources/


In [63]:
import SimpleITK as sitk
import os

def register_and_mask(image_path, template_path, output_path):
    # Load images
    fixed = sitk.ReadImage(template_path, sitk.sitkFloat32)
    moving = sitk.ReadImage(image_path, sitk.sitkFloat32)

    # Basic Rigid Registration
    initial_transform = sitk.CenteredTransformInitializer(fixed, moving, sitk.Euler3DTransform(), sitk.CenteredTransformInitializerFilter.GEOMETRY)
    registration_method = sitk.ImageRegistrationMethod()
    registration_method.SetMetricAsMeanSquares()
    registration_method.SetOptimizerAsRegularStepGradientDescent(learningRate=1.0, minStep=1e-4, numberOfIterations=100)
    registration_method.SetInitialTransform(initial_transform, inPlace=False)
    registration_method.SetInterpolator(sitk.sitkLinear)

    final_transform = registration_method.Execute(fixed, moving)
    resampled = sitk.Resample(moving, fixed, final_transform, sitk.sitkLinear, 0.0, moving.GetPixelID())

    # Save result
    sitk.WriteImage(resampled, output_path)
    print(f"Processed: {os.path.basename(image_path)}")

# Reference template from the unzipped folder
template = "/content/MSLesSeg-2024-main/resources/MNI152_T1_1mm.nii.gz"

print("Starting SimpleITK-based Registration...")
# Example for one subject to verify
if all_files:
    sample = all_files[0]
    register_and_mask(sample['t1'], template, "registered_t1_sample.nii.gz")

Starting SimpleITK-based Registration...
Processed: P64_T1.nii.gz


In [62]:
# Verify FSL installation
import os

fsl_bin_path = '/usr/lib/fsl/5.0'
if os.path.exists(os.path.join(fsl_bin_path, 'flirt')):
    print("✅ FSL installation verified: flirt and bet are available.")
    !flirt -version
else:
    print("❌ FSL binaries not found in /usr/lib/fsl/5.0. Installation may have failed.")

❌ FSL binaries not found in /usr/lib/fsl/5.0. Installation may have failed.


## Preprocessing Functions

In [54]:
from monai.transforms import Resized
import nibabel as nib
import shutil

def collect_dataset(root_path):
    """Collects MSLesSeg data, handling nested directories and extensionless files."""
    data_list = []
    if not os.path.exists(root_path): return []

    for subject in sorted(os.listdir(root_path)):
        subj_path = os.path.join(root_path, subject)
        if not os.path.isdir(subj_path): continue

        files = os.listdir(subj_path)
        item = {"subject": subject}

        def find_file(patterns):
            for f in files:
                if any(p.upper() in f.upper() for p in patterns):
                    full_path = os.path.join(subj_path, f)

                    # If it's a directory (Train set structure), look inside it
                    if os.path.isdir(full_path):
                        sub_files = [sf for sf in os.listdir(full_path) if not sf.startswith('.')]
                        if not sub_files: continue
                        full_path = os.path.join(full_path, sub_files[0])
                        f = sub_files[0]

                    # Handle extensionless files
                    if '.' not in f:
                        temp_path = full_path + '.nii.gz'
                        if not os.path.exists(temp_path):
                             shutil.copy(full_path, temp_path)
                        return temp_path
                    return full_path
            return None

        item["t1"] = find_file(["T1"])
        item["t2"] = find_file(["T2"])
        item["flair"] = find_file(["FLAIR"])
        item["label"] = find_file(["T3", "MASK", "GT"])

        if item["label"] and (item["t1"] or item["t2"] or item["flair"]):
            main_img = item["flair"] or item["t2"] or item["t1"]
            item["flair"] = item["flair"] or main_img
            item["t1"] = item["t1"] or main_img
            item["t2"] = item["t2"] or main_img
            data_list.append(item)

    return data_list

def binarize_label(x):
    return (x > 0.5).astype(np.float32)

def create_transforms():
    return Compose([
        LoadImaged(keys=["flair", "t1", "t2", "label"]),
        EnsureChannelFirstd(keys=["flair", "t1", "t2", "label"]),
        Orientationd(keys=["flair", "t1", "t2", "label"], axcodes="RAS"),
        Spacingd(keys=["flair", "t1", "t2", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "bilinear", "bilinear", "nearest")),
        Resized(keys=["flair", "t1", "t2", "label"], spatial_size=(96, 96, 96), mode=("trilinear", "trilinear", "trilinear", "nearest")),
        Lambdad(keys="label", func=binarize_label),
        CropForegroundd(keys=["flair", "t1", "t2", "label"], source_key="flair"),
        NormalizeIntensityd(keys=["flair", "t1", "t2"], nonzero=True, channel_wise=True),
        ConcatItemsd(keys=["flair", "t1", "t2"], name="image", dim=0),
        DeleteItemsd(keys=["flair", "t1", "t2"]),
        EnsureTyped(keys=["image", "label"]),
    ])

In [47]:
from monai.data import pad_list_data_collate

def get_loaders(data_files, cache_dir, val_frac=0.2):
    random.seed(42)
    random.shuffle(data_files)
    val_size = int(len(data_files) * val_frac)
    train_files, val_files = data_files[val_size:], data_files[:val_size]

    train_ds = PersistentDataset(data=train_files, transform=create_transforms(), cache_dir=cache_dir)
    val_ds = PersistentDataset(data=val_files, transform=create_transforms(), cache_dir=cache_dir)

    return (
        DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=pad_list_data_collate),
        DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=pad_list_data_collate)
    )

## Execution

In [55]:
# Configuration
CACHE_DIR = "/content/cache_mslesseg"

# 1. Collect all valid cases (Train + Test folders combined for maximum data)
train_files = collect_dataset(mslesseg_train_data)
test_files = collect_dataset(mslesseg_test_data)
all_files = train_files + test_files

print(f"Total valid cases found: {len(all_files)} (Train: {len(train_files)}, Test: {len(test_files)})")

# 2. Setup Loaders
if all_files:
    train_loader, val_loader = get_loaders(all_files, CACHE_DIR)
    print("Data loaders successfully initialized.")
else:
    print("Error: No valid cases with labels found in Drive.")

Total valid cases found: 31 (Train: 9, Test: 22)
Data loaders successfully initialized.


In [50]:
# Re-initialize loaders with the new resizing transform
train_loader, val_loader = get_loaders(all_files, CACHE_DIR)
print("Data loaders updated with resizing transform.")

Data loaders updated with resizing transform.


In [48]:
# Re-initialize loaders with the new resizing transform
train_loader, val_loader = get_loaders(all_files, CACHE_DIR)
print("Data loaders updated with resizing transform.")

Data loaders updated with resizing transform.


In [38]:
# Execution
data_files = load_mslesseg_data(mslesseg_train_data)

if data_files:
    transforms = create_mslesseg_transforms()
    val_loader = setup_data_loader(data_files, transforms, CACHE_DIR)
else:
    print("Error: Dataset is empty.")

Searching for data in: /content/drive/MyDrive/mslesseg folder/MSLesSeg Dataset/train...
MSLesSeg cases found: 10
Data loader ready. Train: 8, Val: 2


In [56]:
# Model, Optimizer and Loss Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet(
    spatial_dims=3,
    in_channels=3,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

loss_function = DiceLoss(sigmoid=True)
optimizer = torch.optim.Adam(model.parameters(), 1e-3)
dice_metric = DiceMetric(include_background=False, reduction="mean")

print(f"Model initialized on {device}.")

Model initialized on cuda.


In [57]:
# Training Loop
max_epochs = 50
val_interval = 2
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []

print("Starting Training...")
for epoch in range(max_epochs):
    model.train()
    epoch_loss = 0
    step = 0
    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)

    if (epoch + 1) % val_interval == 0:
        model.eval()
        with torch.no_grad():
            for val_data in val_loader:
                val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
                val_outputs = model(val_inputs)
                val_outputs = (torch.sigmoid(val_outputs) > 0.5).float()
                dice_metric(y_pred=val_outputs, y=val_labels)

            metric = dice_metric.aggregate().item()
            dice_metric.reset()
            metric_values.append(metric)
            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch + 1
                torch.save(model.state_dict(), "best_mslesseg_model.pth")

            print(f"Epoch {epoch+1}: Loss {epoch_loss:.4f}, Val Dice {metric:.4f}")

print(f"Training complete. Best Dice: {best_metric:.4f} at epoch {best_metric_epoch}")

Starting Training...


RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 11 but got size 12 for tensor number 1 in the list.

In [49]:
# Training Loop
max_epochs = 50
val_interval = 2
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []

print("Starting Training...")
for epoch in range(max_epochs):
    model.train()
    epoch_loss = 0
    step = 0
    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)

    if (epoch + 1) % val_interval == 0:
        model.eval()
        with torch.no_grad():
            for val_data in val_loader:
                val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
                val_outputs = model(val_inputs)
                val_outputs = (torch.sigmoid(val_outputs) > 0.5).float()
                dice_metric(y_pred=val_outputs, y=val_labels)

            metric = dice_metric.aggregate().item()
            dice_metric.reset()
            metric_values.append(metric)
            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch + 1
                torch.save(model.state_dict(), "best_mslesseg_model.pth")

            print(f"Epoch {epoch+1}: Loss {epoch_loss:.4f}, Val Dice {metric:.4f}")

print(f"Training complete. Best Dice: {best_metric:.4f} at epoch {best_metric_epoch}")

Starting Training...


RuntimeError: applying transform <monai.transforms.io.dictionary.LoadImaged object at 0x7baf3cf0c5c0>

In [32]:
import pandas as pd

# Collect data for both sets using the updated logic
train_cases = load_mslesseg_data(mslesseg_train_data)
test_cases = load_mslesseg_data(mslesseg_test_data)

print(f"\n--- Found {len(train_cases)} Training Cases ---")
if train_cases:
    display(pd.DataFrame(train_cases).head())

print(f"\n--- Found {len(test_cases)} Testing Cases ---")
if test_cases:
    display(pd.DataFrame(test_cases).head())

Searching for data in: /content/drive/MyDrive/mslesseg folder/MSLesSeg Dataset/train...
MSLesSeg cases found: 10
Searching for data in: /content/drive/MyDrive/mslesseg folder/MSLesSeg Dataset/test...
MSLesSeg cases found: 22

--- Found 10 Training Cases ---


,subject,t1,t2,label,flair
0,P1,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...
1,P12,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...
2,P14,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...
3,P19,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...
4,P2,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...



--- Found 22 Testing Cases ---


,subject,t1,t2,label,flair
0,P54,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...
1,P55,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...
2,P56,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...
3,P57,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...
4,P58,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...,/content/drive/MyDrive/mslesseg folder/MSLesSe...


In [33]:
def audit_dataset(root_path):
    print(f"Auditing: {root_path}")
    all_items = sorted(os.listdir(root_path))
    valid_cases = []
    skipped = []

    for item in all_items:
        path = os.path.join(root_path, item)
        if not os.path.isdir(path):
            skipped.append(f"{item} (Not a folder)")
            continue

        files = os.listdir(path)
        has_t1 = any("T1" in f.upper() for f in files)
        has_t2 = any("T2" in f.upper() for f in files)
        has_label = any(x in " ".join(files).upper() for x in ["T3", "MASK", "GT"])

        if has_t1 and has_label:
            valid_cases.append(item)
        else:
            reasons = []
            if not has_t1: reasons.append("Missing T1")
            if not has_label: reasons.append("Missing Label (T3/MASK/GT)")
            skipped.append(f"{item} ({', '.join(reasons)})")

    print(f"Total items found: {len(all_items)}")
    print(f"Valid cases found: {len(valid_cases)}")
    if skipped:
        print("\nReasons for skipping items:")
        for s in skipped[:15]: # Show first 15 for brevity
            print(f" - {s}")
        if len(skipped) > 15: print(f" ... and {len(skipped)-15} more.")

print("--- TRAIN AUDIT ---")
audit_dataset(mslesseg_train_data)
print("\n--- TEST AUDIT ---")
audit_dataset(mslesseg_test_data)

--- TRAIN AUDIT ---
Auditing: /content/drive/MyDrive/mslesseg folder/MSLesSeg Dataset/train
Total items found: 53
Valid cases found: 10

Reasons for skipping items:
 - P10 (Missing Label (T3/MASK/GT))
 - P11 (Missing Label (T3/MASK/GT))
 - P13 (Missing Label (T3/MASK/GT))
 - P15 (Missing Label (T3/MASK/GT))
 - P16 (Missing Label (T3/MASK/GT))
 - P17 (Missing Label (T3/MASK/GT))
 - P18 (Missing Label (T3/MASK/GT))
 - P21 (Missing Label (T3/MASK/GT))
 - P22 (Missing Label (T3/MASK/GT))
 - P23 (Missing Label (T3/MASK/GT))
 - P24 (Missing Label (T3/MASK/GT))
 - P25 (Missing Label (T3/MASK/GT))
 - P26 (Missing Label (T3/MASK/GT))
 - P27 (Missing Label (T3/MASK/GT))
 - P28 (Missing Label (T3/MASK/GT))
 ... and 28 more.

--- TEST AUDIT ---
Auditing: /content/drive/MyDrive/mslesseg folder/MSLesSeg Dataset/test
Total items found: 22
Valid cases found: 22


In [34]:
readme_path = "/content/drive/MyDrive/mslesseg folder/MSLesSeg-2024-main/README.md"

try:
    with open(readme_path, 'r') as f:
        content = f.read()
    print("--- README CONTENT ---")
    print(content)
except FileNotFoundError:
    print(f"File not found at: {readme_path}")
except Exception as e:
    print(f"An error occurred: {e}")

--- README CONTENT ---
# MSLesSeg-2024: baseline and benchmarking of a new Multiple Sclerosis Lesion Segmentation dataset 

This is the official implementation of the preprocessing applied to the MSLesSeg-2024 dataset, described in the paper:  
["MSLesSeg-2024: baseline and benchmarking of a new Multiple Sclerosis Lesion Segmentation dataset "]()

## Abstract
This paper presents MSLesSeg-2024, a new, publicly accessible MRI dataset designed to advance research in Multiple Sclerosis (MS) lesion segmentation. The dataset comprises 115 scans of 75 patients including T1, T2 and FLAIR sequences, along with supplementary clinical data collected across different sources. Expert-validated annotations provide high-quality lesion segmentation labels, establishing a reliable human-labeled dataset for benchmarking. Part of the dataset was shared with expert scientists with the aim to compare the last automatic AI-based image segmentation solutions with an expert-biased handmade segmentation. In ad

In [35]:
import os

def debug_folder_structure(path):
    print(f"Checking path: {path}")
    if not os.path.exists(path):
        print("Path does not exist.")
        return

    subjects = sorted(os.listdir(path))
    print(f"Found {len(subjects)} items in folder.")
    if subjects:
        first_subj = subjects[0]
        subj_path = os.path.join(path, first_subj)
        print(f"\nContent of first subject folder ({first_subj}):")
        if os.path.isdir(subj_path):
            print(os.listdir(subj_path))
        else:
            print("This item is a file, not a directory.")

print("--- Training Set Debug ---")
debug_folder_structure(mslesseg_train_data)
print("\n--- Testing Set Debug ---")
debug_folder_structure(mslesseg_test_data)

--- Training Set Debug ---
Checking path: /content/drive/MyDrive/mslesseg folder/MSLesSeg Dataset/train
Found 53 items in folder.

Content of first subject folder (P1):
['T1', 'T2', 'T3']

--- Testing Set Debug ---
Checking path: /content/drive/MyDrive/mslesseg folder/MSLesSeg Dataset/test
Found 22 items in folder.

Content of first subject folder (P54):
['P54_T1.nii.gz', 'P54_T2.nii.gz', 'P54_MASK.nii.gz', 'P54_FLAIR.nii.gz']
